# 13-Qubit Pauli-Graph Quantum Hadamard Edge Detection (QHED) Reference

This notebook provides an independently specified, gate-level 13-qubit reference implementation of the **FRQI + Pauli-graph Quantum Hadamard Edge Detection (QHED)** construction from Section VII of:

> Syed Emad Uddin Shubha and Tasnuva Farheen, *"Distinguishing Coherent Crosstalk from Calibration Drift via Pauli-Transfer Signatures and Quantum Edge Detection,"* IEEE HOST 2027 (`arXiv:2609.40283`).

### Scope and Distinction from Precursor Material
- **`NEQR_FRQI.ipynb`** in this repository is earlier precursor material demonstrating general FRQI and NEQR image encoding, decoding, and measurement-based image reconstruction.
- **This notebook (`Pauli_Graph_QHED_Reference.ipynb`)** implements the 13-qubit operator-space Pauli-graph edge-readout circuit ($11$ address + $1$ pixel + $1$ auxiliary qubit, $8,192$ complex statevector amplitudes) and verifies it against two independent classical baselines across five explicitly specified three-qubit unitary channels. It validates the circuit identity on independently specified examples rather than replaying the original manuscript's Figure 10 sweep.


In [1]:
import platform
import numpy as np
import pauli_graph_qhed as qhed

labels = qhed.low_weight_pauli_labels(n_qubits=3, max_weight=2)
print(f"Python: {platform.python_version()} | NumPy: {np.__version__}")
print(f"Physical qubits n = 3, max Pauli weight k = 2 -> K = {len(labels)} low-weight Paulis")
print(f"Feature coordinates K^2 = {len(labels)**2}, padded to N_pad = 2048 (11 address + 1 pixel + 1 aux = 13 qubits)")
print("First 10 ordered Pauli labels:", labels[:10])


Python: 3.13.3 | NumPy: 2.3.5
Physical qubits n = 3, max Pauli weight k = 2 -> K = 37 low-weight Paulis
Feature coordinates K^2 = 1369, padded to N_pad = 2048 (11 address + 1 pixel + 1 aux = 13 qubits)
First 10 ordered Pauli labels: ('III', 'IIX', 'IIY', 'IIZ', 'IXI', 'IXX', 'IXY', 'IXZ', 'IYI', 'IYX')


## 1. Pauli Graphs and Deterministic Matching Decompositions

Each pixel $j \in \{0, \dots, 1368\}$ indexes an ordered pair $(P_\alpha, P_\beta)$ in the $37 \times 37$ low-weight Pauli transfer matrix (PTM) block:
- **Hamming-1 graph ($G_H$):** connects coordinates where one Pauli label is unchanged and the other differs in exactly one tensor factor.
- **Anticommuting-factor graph ($G_C$):** restricts $G_H$ to pairs whose differing single-qubit tensor factors anticommute.


In [2]:
graphs = qhed.build_pauli_graphs(labels)
for name, spec in graphs.items():
    cx_count = sum(2 * ((u ^ v).bit_count()) - 1 for u, v in spec.edges)
    print(
        f"{name:22s} | vertices={spec.n_vertices:4d} | undirected_edges={len(spec.edges):5d} "
        f"| matchings={len(spec.matchings):2d} | 11-controlled-X={cx_count:5d}"
    )


hamming_1              | vertices=1369 | undirected_edges= 9324 | matchings=18 | 11-controlled-X=62400
anticommuting_factor   | vertices=1369 | undirected_edges= 4662 | matchings= 9 | 11-controlled-X=30638


## 2. Analytical & Gate-Level Walkthrough: $U = e^{-i(0.23)ZXI/2}$

For a nonnegative, nonzero feature vector $S$ normalized by $\widetilde{S}_j = S_j / \max_k S_k$, the FRQI loader applies $R_y(2\arcsin\widetilde{S}_j)$ conditioned on each 11-bit address $|j\rangle$. Conditioning on pixel outcome $|1\rangle$ prepares $a_j = S_j / \|S\|_2$ with probability $p_{\mathrm{load}} = \sum_j \widetilde{S}_j^2 / N_{\mathrm{pad}}$.

For each matching $M_\ell$, an address involution $D_\ell$ swaps paired addresses along binary Gray paths ($2d - 1$ controlled-$X$ primitives for Hamming distance $d$). Applying $H_{\mathrm{aux}}, \text{controlled-}D_\ell, H_{\mathrm{aux}}$ yields auxiliary-$1$ probability $P_\ell = \frac{1}{2}\sum_{\{u,v\}\in M_\ell} |a_u - a_v|^2$, so:
$$\mathcal{A}_{\mathrm{edge}}^G(S) = \frac{1}{2}\sum_\ell P_\ell = \frac{S^{\mathsf{T}} L_G S}{4 S^{\mathsf{T}} S}.$$


In [3]:
unitaries = qhed.reference_unitary_examples()
ptm_zx = qhed.low_weight_ptm(unitaries["zx_crosstalk"]["unitary"], labels)
feat_zx = qhed.extract_feature_images(ptm_zx, labels)["cross_weight_antisymmetric"]

print(f"Nonzero pixels in ZX cross-weight antisymmetric image: {int(np.count_nonzero(feat_zx))}")
for g_name, g_spec in graphs.items():
    cl_edge = qhed.classical_edge_sum_score(feat_zx, g_spec.edges)
    cl_lap = qhed.classical_laplacian_score(feat_zx, g_spec.laplacian)
    res = qhed.execute_pauli_graph_qhed_circuit(feat_zx, g_spec, n_address_qubits=11)
    print(
        f"{g_name:22s} | classical={cl_edge:.15f} | laplacian={cl_lap:.15f} "
        f"| circuit={res.edge_score:.15f} | |diff|={abs(res.edge_score - cl_edge):.3e}"
    )


Nonzero pixels in ZX cross-weight antisymmetric image: 8
hamming_1              | classical=3.750000000000000 | laplacian=3.750000000000000 | circuit=3.749999999999999 | |diff|=8.882e-16
anticommuting_factor   | classical=1.500000000000000 | laplacian=1.500000000000000 | circuit=1.500000000000000 | |diff|=4.441e-16


In [4]:
for g_name, g_spec in graphs.items():
    res = qhed.execute_pauli_graph_qhed_circuit(feat_zx, g_spec, n_address_qubits=11)
    print(
        f"{g_name:22s} | width={res.n_qubits} | loader_H={res.loader_hadamards} | loader_mcry={res.loader_controlled_ry} "
        f"| p_load={res.p_load:.8f} (1/{round(1.0 / res.p_load)}) | matchings={res.matching_circuits} "
        f"| aux_H={res.aux_hadamards} | 11-ctrl-X={res.controlled_x_primitives} "
        f"| expected_attempts={res.expected_loader_attempts:.1f}"
    )


hamming_1              | width=13 | loader_H=11 | loader_mcry=8 | p_load=0.00390625 (1/256) | matchings=18 | aux_H=36 | 11-ctrl-X=62400 | expected_attempts=4608.0
anticommuting_factor   | width=13 | loader_H=11 | loader_mcry=8 | p_load=0.00390625 (1/256) | matchings=9 | aux_H=18 | 11-ctrl-X=30638 | expected_attempts=2304.0


## 3. Full Five-Unitary Benchmark Suite (28 Nonzero + 2 Zero-Image Cases)

We evaluate five 3-qubit unitaries (local drift, $ZX$, $YX$, $ZZ$, and a noncommuting combination) across three low-weight PTM feature images and both Pauli graphs. Under local coherent drift, the cross-weight antisymmetric feature vanishes identically by Proposition 1, producing the two zero-image cases ($p_{\mathrm{load}} = 0$).


In [5]:
benchmark = qhed.run_reference_benchmark_suite()
print(f"Total cases: {benchmark['total_cases']} ({benchmark['nonzero_cases']} nonzero, {benchmark['zero_image_cases']} zero-image)")
print(f"Max |circuit - classical edge sum| : {benchmark['max_abs_diff_circuit_vs_edge_sum']:.16e}")
print(f"Max |circuit - classical Laplacian|: {benchmark['max_abs_diff_circuit_vs_laplacian']:.16e}")


Total cases: 30 (28 nonzero, 2 zero-image)
Max |circuit - classical edge sum| : 1.7763568394002505e-15
Max |circuit - classical Laplacian|: 1.3322676295501878e-15


In [6]:
print(f"{'Unitary':18s} | {'Feature':26s} | {'Graph':20s} | {'Circuit Score':>16s} | {'|Diff|':>10s}")
print("-" * 102)
for row in benchmark["cases"]:
    if row["circuit_score"] is None:
        print(f"{row['unitary']:18s} | {row['feature']:26s} | {row['graph']:20s} | {'undefined (S=0)':>16s} | {'n/a':>10s}")
    else:
        print(
            f"{row['unitary']:18s} | {row['feature']:26s} | {row['graph']:20s} "
            f"| {row['circuit_score']:16.12f} | {row['abs_diff_edge_sum']:10.2e}"
        )


Unitary            | Feature                    | Graph                |    Circuit Score |     |Diff|
------------------------------------------------------------------------------------------------------
local_drift        | cross_weight_antisymmetric | hamming_1            |  undefined (S=0) |        n/a
local_drift        | cross_weight_antisymmetric | anticommuting_factor |  undefined (S=0) |        n/a
local_drift        | low_weight_antisymmetric   | hamming_1            |   3.216791378751 |   8.88e-16
local_drift        | low_weight_antisymmetric   | anticommuting_factor |   1.855472414166 |   0.00e+00
local_drift        | off_diagonal_residual      | hamming_1            |   3.136592676857 |   4.44e-16
local_drift        | off_diagonal_residual      | anticommuting_factor |   1.777373902945 |   4.44e-16
zx_crosstalk       | cross_weight_antisymmetric | hamming_1            |   3.750000000000 |   8.88e-16
zx_crosstalk       | cross_weight_antisymmetric | anticommuting_factor | 

## 4. Boundary Conditions and Optional Qiskit Small-Width Cross-Check

We verify two distinct boundary behaviors:
1. **$S = 0$ (zero feature image):** pixel-$1$ postselection has probability $p_{\mathrm{load}} = 0$, so the normalized score is undefined.
2. **Constant $S > 0$ across all $1,369$ graph vertices:** every graph edge connects equal amplitudes, so $\mathcal{A}_{\mathrm{edge}}^G(S) = 0$ even though padded addresses $1369 \dots 2047$ hold zero amplitude (because matching involutions fix all padded addresses).


In [7]:
zero_vec = np.zeros(len(labels) ** 2, dtype=np.float64)
zero_res = qhed.execute_pauli_graph_qhed_circuit(zero_vec, graphs["hamming_1"])
print("Zero feature S = 0 -> p_load =", zero_res.p_load, "| edge_score =", zero_res.edge_score)

const_vec = np.full(len(labels) ** 2, 0.5, dtype=np.float64)
for g_name, g_spec in graphs.items():
    c_res = qhed.execute_pauli_graph_qhed_circuit(const_vec, g_spec)
    c_cl = qhed.classical_edge_sum_score(const_vec, g_spec.edges)
    print(f"Constant feature S > 0 ({g_name:20s}) -> classical={c_cl:.16f} | circuit={c_res.edge_score:.16e}")


Zero feature S = 0 -> p_load = 0.0 | edge_score = None
Constant feature S > 0 (hamming_1           ) -> classical=0.0000000000000000 | circuit=0.0000000000000000e+00
Constant feature S > 0 (anticommuting_factor) -> classical=0.0000000000000000 | circuit=0.0000000000000000e+00


In [8]:
qk_info = qhed.qiskit_small_width_crosscheck()
if qk_info is None:
    print("Optional Qiskit package not installed; core NumPy reference verified.")
else:
    print(
        f"Optional 4-qubit Qiskit {qk_info['qiskit_version']} statevector check: "
        f"loader max |Δψ| = {qk_info['loader_max_amplitude_diff']:.3e}, "
        f"full circuit max |Δψ| = {qk_info['full_circuit_max_amplitude_diff']:.3e}"
    )


Optional 4-qubit Qiskit 2.1.0 statevector check: loader max |Δψ| = 1.110e-16, full circuit max |Δψ| = 1.110e-16
